# Client SDK (040) Demo

Demonstrate the Python client SDK using a one-shot local socket server.


In [8]:
%load_ext autoreload
%autoreload 2
import json
import socket
import tempfile
import threading
from pathlib import Path

from ciphercache.client import CipherClient, CipherClientConfig
from ciphercache.ipc.framing import encode_message

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
data_dir = Path(tempfile.mkdtemp(prefix="ciphercache-sdk-demo-"))
socket_path = data_dir / "ciphercached.sock"
tickets_dir = data_dir / "tickets"
tickets_dir.mkdir(parents=True, exist_ok=True)
(tickets_dir / "default.ticket").write_text("demo-token", encoding="utf-8")
data_dir


PosixPath('/var/folders/0t/w9l_5c597rdglh2kbffq18sr0000gn/T/ciphercache-sdk-demo-exgvp_fr')

In [10]:
def serve_once(response_envelope):
    """Simulates the daemon by running a thread, closes after client.ping()"""
    if socket_path.exists():
        socket_path.unlink()
    def run():
        print("Running simulated daemon")
        listener = socket.socket(socket.AF_UNIX, socket.SOCK_STREAM)
        print("Listening on ", listener)
        try:
            listener.bind(str(socket_path))
            listener.listen(1)
            conn, _ = listener.accept()
            print("Accepted")
            try:
                length_prefix = conn.recv(4)
                length = int.from_bytes(length_prefix, "big")
                _ = conn.recv(length)
                print("received: ", _)
                conn.sendall(encode_message(response_envelope))
                print("Sent ", response_envelope)
            finally:
                conn.close()
        finally:
            listener.close()
            if socket_path.exists():
                socket_path.unlink()
    thread = threading.Thread(target=run, daemon=True)
    thread.start()
    return thread

client = CipherClient(config=CipherClientConfig(data_dir=data_dir))

response = {
    "version": "v0",
    "id": "ping",
    "type": "response",
    "op": "ping",
    "payload": {"ok": True},
}
thread = serve_once(response)
client.ping()


True

In [11]:
response = {
    "version": "v0",
    "id": "status",
    "type": "response",
    "op": "status",
    "payload": {"locked": False, "ttl_remaining_seconds": 120},
}
thread = serve_once(response)
client.status()


Status(locked=False, ttl_remaining_seconds=120)

In [12]:
response = {
    "version": "v0",
    "id": "secret",
    "type": "response",
    "op": "get_secret",
    "payload": {"secret": {"api_key": "demo"}},
}
thread = serve_once(response)
client.get_secret("service/api")


{'api_key': '<*redacted*>'}

In [6]:
# Requires a running daemon in demo mode (e.g. `uv run python scripts/run_daemon.py --demo --db-path ../testdata/demopasswords.kdbx`).
# This cell performs a full client lifecycle against the live daemon.
from ciphercache.client import CipherClient, CipherClientConfig

client = CipherClient(config=CipherClientConfig())
ticket_path = client.client_init("notebook_demo")
client.config.ticket_path = ticket_path
client.load_ticket()
client.status()
client.get_secret("service/api")["value"].reveal()
